**To-dos (model itself):**
*   Incorporate variables for has_media and engagement_type into the model and instructions using langchain

*   Instruct model to turn keywords from prompts into queries so it isn't searching the entire database every time (this should make it run faster)

* Refine user expertise levels to be more specific for our application


**To-dos (overall):**

* Integrate into the streamlit app


**Questions:**

*   If user doesn't specify a variable in the UI do we want the LLM to infer it anyways when conducting queries? Or specifically instruct the model to treat a nonspecified variable as "All"?

* Do we want to instruct the model to answer questions based ONLY on what it can access in our statistics, or can it rely on its own background knowledge to fill in gaps?





In [14]:
# To use this demo, upload the rag_model_rules_FINAL.parquet and rag_predictions_FINAL.parquet
# to your notebook runtime.

# Run this in a Colab cell to install the basics
!pip install -q langchain langchain-google-genai

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Set your API key for the session
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6Khu1XZfKsdkjFdyVLQTwG_7Su0B7rV5P7SunfnvVoqoA"

# Initialize the model
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

# Draft a prompt specific to your hackathon goals
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant designed to analyze Reddit behavior and virality."),
    ("user", "{user_input}")
])

# Chain them together
chain = prompt | llm

# Test how it responds
response = chain.invoke({"user_input": "What factors make a post go viral on r/learnprogramming?"})
print(response.content)

# 4. Define your user profiles/personas
PERSONA_INSTRUCTIONS = {
    "Beginner": (
        "Explain concepts simply, using plain language that someone without professional or academic experience with statistics or marketing would understand. Avoid technical jargon or heavy statistical terms. "
        "Keep the tone encouraging, accessible, and high-level."
    ),
    "Professional Marketer": (
        "Explain concepts in a way that a marketing professional without advanced mathematical expertise would understand."
        "Focus on actionable takeaways, audience engagement, conversion potential, and growth trends. "
        "Use marketing terminology like CTA, conversion, organic reach, virality coefficient, and positioning."
    ),
    "Statistical Expert": (
        "Focus on data validity, sample sizes, probability distributions, variance, correlation vs causation, "
        "and metrics like p-values or standard deviations. Be precise, analytical, and objective."
    )
}

# 5. Mock the upcoming DuckDB/Parquet query function
def mock_parquet_lookup(query: str) -> str:
    """
    Simulates retrieving relevant rows from your exported Parquet tables.
    """
    print(f"🔍 [System] Searching Parquet tables for keywords related to: '{query}'...")
    return (
        "- Top Subreddit Match: r/dataisbeautiful (Avg Upvotes: 4.2k, Engagement Rate: 12.4%)\n"
        "- Peak Posting Time: Tuesday 14:00 UTC\n"
        "- Virality Score Correlation: Title Sentiment (+0.34), Image Included (+0.58)"
    )

# 6. Build the LangChain Prompt and Pipeline
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Reddit Analytics Assistant.\n"
     "Target Audience Level: {expertise_level}\n"
     "Persona Style: {persona_instruction}\n\n"
     "Relevant Data Context:\n{retrieved_data}\n\n"
     "Answer the user's question adhering strictly to the required audience level and persona style."
    ),
    ("user", "{user_input}")
])

# Chain: Prompt -> Model -> String Output
chain = prompt | llm | StrOutputParser()

def run_demo(user_input: str, expertise_level: str):
    """Executes the backend pipeline and prints the output."""
    print(f"\n👤 [User Profile]: {expertise_level}")
    print(f"💬 [Question]: {user_input}\n")

    # Step A: Retrieve Data Context
    data_context = mock_parquet_lookup(user_input)

    # Step B: Select Persona
    persona_style = PERSONA_INSTRUCTIONS.get(expertise_level, PERSONA_INSTRUCTIONS["Beginner"])

    # Step C: Generate Response
    print("🤖 [Generating Response...]\n")
    response = chain.invoke({
        "expertise_level": expertise_level,
        "persona_instruction": persona_style,
        "retrieved_data": data_context,
        "user_input": user_input
    })

    print("-" * 50)
    print(response)
    print("-" * 50)

# ==========================================
# 7. DEMO TESTING AREA
# Change these variables to test different outputs during your meeting
# ==========================================

TEST_QUESTION = "Why do posts with images perform better on Reddit?"
CURRENT_PROFILE = "Statistical Expert" # Try "Beginner" or "Professional Marketer"

# Run the pipeline
run_demo(TEST_QUESTION, CURRENT_PROFILE)

[{'type': 'text', 'text': 'To understand virality on **r/learnprogramming** (a community of over 3.7 million members), you have to understand its demographic. The subreddit is populated by self-taught developers, bootcamp grads, CS students, and experienced mentors. \n\nBecause r/learnprogramming **strictly bans memes, low-effort images, and direct self-promotion**, virality here is driven purely by text-based value, emotional resonance, and community discussion.\n\nHere is an analysis of the key factors and post archetypes that drive virality on r/learnprogramming.\n\n---\n\n### 1. The Core Viral Archetypes\nAlmost every post that reaches the top of r/learnprogramming fits into one of five distinct categories:\n\n#### A. "The Hope Merchant" (Success Stories)\nThese are stories of career transitions, usually written by self-taught developers or older career-changers who landed their first job. \n*   **Why they go viral:** They offer light at the end of the tunnel for discouraged learne

In [17]:
# Demo Scenario 2: Marketing Focus
# The LLM will shift to focus on growth, audience reach, and actionable strategy.

demo_2_question = "When is the absolute best time to post if we want to maximize our organic reach?"
demo_2_profile = "Professional Marketer"

run_demo(demo_2_question, demo_2_profile)


👤 [User Profile]: Professional Marketer
💬 [Question]: When is the absolute best time to post if we want to maximize our organic reach?

🔍 [System] Searching Parquet tables for keywords related to: 'When is the absolute best time to post if we want to maximize our organic reach?'...
🤖 [Generating Response...]

--------------------------------------------------
To maximize your organic reach and give your content the highest possible chance of trending, the absolute best time to post is **Tuesday at 14:00 UTC**. 

In marketing terms, this is your "golden window." Tuesday at 14:00 UTC (9:00 AM EST / 6:00 AM PST) perfectly captures the mid-week, mid-day scroll across major global markets. Posting at this precise moment ensures your content hits the platform just as user activity surges, giving you the critical initial traction needed to trigger Reddit’s algorithms and push your post to the top of users' feeds.

However, timing is only the first step. To truly optimize your **virality coef

In [15]:
# Demo Scenario 3: Beginner Focus
# The LLM will avoid jargon and explain the mock data using simple analogies.

demo_3_question = "What does it mean when a post has a high engagement rate?"
demo_3_profile = "Beginner"

run_demo(demo_3_question, demo_3_profile)


👤 [User Profile]: Beginner
💬 [Question]: What does it mean when a post has a high engagement rate?

🔍 [System] Searching Parquet tables for keywords related to: 'What does it mean when a post has a high engagement rate?'...
🤖 [Generating Response...]

--------------------------------------------------
Think of a high engagement rate as a **digital round of applause**! 

When you post something online, people usually do one of two things: they either glance at it and keep scrolling, or they stop and actually do something—like upvoting, leaving a comment, or sharing it with a friend. 

A **high engagement rate** simply means that a large percentage of the people who saw your post decided to stop, get involved, and join the conversation, rather than just passing by. It shows that your post really connected with people and made them want to interact with you!

### What does this look like in real life?
Let's look at a popular community on Reddit called **r/dataisbeautiful** (a place where

In [18]:
# Demo Scenario 3: Beginner Focus
# The LLM will avoid jargon and explain the mock data using simple analogies.

demo_4_question = "I want to post a picture of a sandwich. What subreddit should I post it in to get the most attention?"
demo_4_profile = "Beginner"

run_demo(demo_4_question, demo_4_profile)


👤 [User Profile]: Beginner
💬 [Question]: I want to post a picture of a sandwich. What subreddit should I post it in to get the most attention?

🔍 [System] Searching Parquet tables for keywords related to: 'I want to post a picture of a sandwich. What subreddit should I post it in to get the most attention?'...
🤖 [Generating Response...]

--------------------------------------------------
If you want your sandwich to get the absolute most attention possible, our data points to a surprisingly fun and popular community: **r/dataisbeautiful**! 

Now, you might be wondering, *"A sandwich in a data community?"* Yes! The users in this group absolutely love beautiful, creative visual breakdowns. If you take a picture of your sandwich and make a simple, colorful diagram pointing out all the delicious layers (like the bread, the cheese, the secret sauce, and the veggies), they will love it. 

On average, posts in this community get around **4,200 upvotes**, and about **12% of the people who see

In [19]:
demo_5_question = "What are the most influential factors affecting virality for a question post about fashion and beauty?"
demo_5_profile = "Statistical Expert"

run_demo(demo_5_question, demo_5_profile)


👤 [User Profile]: Statistical Expert
💬 [Question]: What are the most influential factors affecting virality for a question post about fashion and beauty?

🔍 [System] Searching Parquet tables for keywords related to: 'What are the most influential factors affecting virality for a question post about fashion and beauty?'...
🤖 [Generating Response...]

--------------------------------------------------
To model the virality of a question-based post within the fashion and beauty domain, we must treat "virality" as a highly right-skewed dependent variable ($Y$), typically conforming to a power-law or log-normal distribution ($Y \sim \text{Log-Normal}(\mu, \sigma^2)$). 

An empirical analysis of subreddit performance metrics and user interaction vectors reveals that the variance in post propagation is driven by several primary independent variables. Below is a rigorous decomposition of these factors, utilizing correlation coefficients, temporal distribution parameters, and benchmark engagem